# UrbanPulse — Notebook 02: Feature Engineering

Bible §5 Notebook 02. Logic lives in `features.py`; this notebook is the narrative layer. **Produces:** `data/features.parquet` + `data/feature_norms.json`.

**Unit note:** speeds in `cleaned.parquet` are already km/h (lanes 1-5), so the Bible's `÷10` in the feature formulas is NOT re-applied here.

**Target note (decision #11):** the literal `queue>400` gives 0.55% positive (the Bible's 13% was Link 36's *per-link* rate). We use `occup>0.5 AND queue>238` → ~13.1% dataset-wide.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))
import features as feat, io_utils, config
import pandas as pd; pd.set_option('display.max_columns', 40)

## 1. Load cleaned data

In [2]:
df = io_utils.load_parquet(config.CLEANED_PARQUET); df.shape

(266112, 41)

## 2. Group A — cross-lane aggregates

In [3]:
feat.group_a_aggregates(df).describe().round(2)

,total_vehs,mean_speed_kmh,mean_queue_s,max_queue_s,mean_occup,max_occup,lane_active_count
count,266112.00,266112.00,266112.00,266112.00,266112.00,266112.00,266112.00
mean,743.12,16.78,217.96,375.44,0.33,0.53,5.15
std,487.50,5.05,77.83,135.02,0.23,0.32,0.58
min,30.00,3.94,16.94,50.46,0.00,0.01,3.00
25%,363.00,12.42,160.57,270.37,0.13,0.20,5.00
50%,672.00,16.66,211.56,358.21,0.30,0.53,5.00
75%,1044.00,20.97,275.15,469.24,0.49,0.81,6.00
max,4037.00,28.60,892.32,1656.59,1.00,1.00,6.00


## 3. Group B — speed-quality (stop-go) features

In [4]:
feat.group_b_speed_quality(df).describe().round(2)

,speed_div_L1,mean_speed_div,speed_var_across_lanes
count,266112.00,266112.00,266112.00
mean,1.79,2.29,52.77
std,1.37,0.88,43.71
min,0.00,0.01,0.03
25%,0.76,1.61,21.79
50%,1.50,2.16,40.69
75%,2.38,2.89,67.21
max,9.21,5.96,316.33


## 4. Group C — time features

In [5]:
feat.group_c_time(df).head()

,is_am_peak,is_pm_peak,is_weekend,minute_of_day,sin_hour,cos_hour
0,0,0,0,0,0.0,1.0
1,0,0,0,5,0.0,1.0
2,0,0,0,10,0.0,1.0
3,0,0,0,15,0.0,1.0
4,0,0,0,20,0.0,1.0


## 5. Group D — KPIs (congestion_index, road_health_score)

In [6]:
a = feat.group_a_aggregates(df)
kpis, norms = feat.group_d_kpis(a)
print('norm constants:', norms)
kpis.describe().round(3)

norm constants: {'queue_min': 16.94333333333333, 'queue_max': 892.3233333333334, 'speed_min': 3.936, 'speed_max': 28.601999999999997}


,congestion_index,road_health_score
count,266112.000,266112.000
mean,0.334,66.601
std,0.080,8.013
min,0.162,20.962
25%,0.268,61.125
50%,0.313,68.737
75%,0.389,73.169
max,0.790,83.844


## 6. Build full feature set + target

In [7]:
features = feat.build_features(df)
print('shape:', features.shape)
features.head()

shape: (266112, 27)


,LINK_ID,date,day_number,day_of_week,hour,total_vehs,mean_speed_kmh,mean_queue_s,max_queue_s,mean_occup,max_occup,lane_active_count,speed_div_L1,mean_speed_div,speed_var_across_lanes,is_am_peak,is_pm_peak,is_weekend,minute_of_day,sin_hour,cos_hour,congestion_index,road_health_score,lane6_active,lane4_stalled,lane5_stalled,congested
0,1,2024-07-01 00:00:00,1,0,0,1075,15.6132,174.901667,247.13,0.76546,1.0,6,0.999,2.4538,0.340033,0,0,0,0,0.0,1.0,0.500987,49.901329,1,0,0,0
1,1,2024-07-01 00:05:00,1,0,0,1113,16.2060,189.220000,233.93,0.65830,1.0,6,0.776,2.5618,0.978244,0,0,0,5,0.0,1.0,0.457839,54.216072,1,0,0,0
2,1,2024-07-01 00:10:00,1,0,0,1099,16.4034,191.550000,254.97,0.63462,1.0,6,0.971,2.4318,0.912731,0,0,0,10,0.0,1.0,0.447298,55.270185,1,0,0,0
3,1,2024-07-01 00:15:00,1,0,0,966,15.5170,171.246667,247.34,0.67478,1.0,6,0.941,2.5056,0.536772,0,0,0,15,0.0,1.0,0.464228,53.577163,1,0,0,0
4,1,2024-07-01 00:20:00,1,0,0,1062,15.7750,192.108333,248.48,0.76886,1.0,6,0.767,2.4574,0.487150,0,0,0,20,0.0,1.0,0.507586,49.241352,1,0,0,0


## 7. B2 gate — no NaN, ~13% positive

In [8]:
import json
print(json.dumps(feat.feature_summary(features), indent=2))

{
  "rows": 266112,
  "n_features": 21,
  "nan_cells": 0,
  "target_positive_rate": 0.13115906084656084
}


## 8. Class balance

In [9]:
features['congested'].value_counts(normalize=True).round(4)

congested
0    0.8688
1    0.1312
Name: proportion, dtype: float64

## 9. Save

In [10]:
io_utils.save_parquet(features, config.FEATURES_PARQUET)
print('saved', config.FEATURES_PARQUET)

saved /home/claude/urbanpulse/data/features.parquet
